In [2]:
%pip install gmsh 


  Using cached gmsh-4.15.2-py2.py3-none-macosx_12_0_arm64.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 6.6 MB/s  0:00:05 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gmsh
import sys

In [ ]:
# Initialize gmsh:
gmsh.initialize()

# cube points:
lc = 1e-1
point1 = gmsh.model.geo.add_point(0, 0, 0, lc)
point2 = gmsh.model.geo.add_point(1, 0, 0, lc)
point3 = gmsh.model.geo.add_point(1, 1, 0, lc)
point4 = gmsh.model.geo.add_point(0, 1, 0, lc)
point5 = gmsh.model.geo.add_point(0, 1, 1, lc)
point6 = gmsh.model.geo.add_point(0, 0, 1, lc)
point7 = gmsh.model.geo.add_point(1, 0, 1, lc)
point8 = gmsh.model.geo.add_point(1, 1, 1, lc)


# Edge of cube:
line1 = gmsh.model.geo.add_line(point1, point2)
line2 = gmsh.model.geo.add_line(point2, point3)
line3 = gmsh.model.geo.add_line(point3, point4)
line4 = gmsh.model.geo.add_line(point4, point1)
line5 = gmsh.model.geo.add_line(point5, point6)
line6 = gmsh.model.geo.add_line(point6, point7)
line7 = gmsh.model.geo.add_line(point7, point8)
line8 = gmsh.model.geo.add_line(point8, point5)
line9 = gmsh.model.geo.add_line(point4, point5)
line10 = gmsh.model.geo.add_line(point6, point1)
line11 = gmsh.model.geo.add_line(point7, point2)
line12 = gmsh.model.geo.add_line(point3, point8)
face1 = gmsh.model.geo.add_curve_loop([line1, line2, line3, line4])
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.1)
gmsh.option.setNumber("Mesh.MeshSizeMin", 0.05)
gmsh.model.geo.add_plane_surface([face1])
# Create the relevant Gmsh data structures 
# from Gmsh model.
gmsh.model.geo.synchronize()

# Generate mesh:
gmsh.model.mesh.generate()

# Write mesh data:
gmsh.write("GFG.msh")

# Creates  graphical user interface
if 'close' not in sys.argv:
    gmsh.fltk.run()

# It finalize the Gmsh API
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.000708334s, CPU 0.000989s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00313379s, CPU 0.003074s)
Info    : Meshing 3D...
Info    : Done meshing 3D (Wall 1.11251e-05s, CPU 3e-06s)
Info    : 218 nodes 370 elements
Info    : Writing 'GFG.msh'...
Info    : Done writing 'GFG.msh'
-------------------------------------------------------
Version       : 4.15.2
License       : GNU General Public L

# Sanity check pass
given a .msh file, we need to verify that the generated mesh passes a couple basic tests, which include:
1. the number of nodes must be greater than 0 
2. the number of elements must be greater than 0 
3. boundary nodes should satisfy expected geometric tests within a numerical tolerance? 

In [3]:
import json
import gmsh
import mesh_check  # check_mesh(node_tags, coords, elements, boundary, expected_bounds) -> report dict

gmsh.initialize()
gmsh.open("GFG.msh")
report = mesh_check.check_mesh(
    **mesh_check.extract_gmsh(),
    expected_bounds=([0, 0, 0], [1, 1, 1]),
)
gmsh.finalize()

print("PASS" if report["pass"] else "FAIL")
for name, c in report["checks"].items():
    print(f"  {'ok ' if c['pass'] else 'BAD'} {name}: {json.dumps({k: v for k, v in c.items() if k != 'pass'})}")

Info    : Reading 'GFG.msh'...
Info    : 21 entities
Info    : 218 nodes
Info    : 370 elements
Info    : Done reading 'GFG.msh'
PASS
  ok  node_count: {"count": 218, "coord_rows": 218, "duplicate_tags": 0}
  ok  element_count: {"total": 370, "domain": 242, "by_type": {"1": 120, "2": 242, "15": 8}}
  ok  coordinate_bounds: {"min": [0.0, 0.0, 0.0], "max": [1.0, 1.0, 1.0]}
  ok  connectivity: {"missing_nodes": 0, "missing_sample": [], "orphan_nodes": 0, "orphan_sample": [], "degenerate_elements": 0}
  ok  boundary_tags: {"skipped": true}


In [ ]:
import sys

import gmsh

# Both layers in one gmsh window: the mesh nodes as a point cloud sitting on the
# triangulated surface. Everything here is a display option -- nothing is regenerated.
gmsh.initialize()
gmsh.open("GFG.msh")

DISPLAY = {
    "Mesh.Points": 1,         # the point cloud: one dot per mesh node   (default 0, hidden)
    "Mesh.PointSize": 5,
    "Mesh.PointType": 1,      # 1 = shaded spheres, 0 = flat dots
    "Mesh.SurfaceEdges": 1,   # the mesh: triangle wireframe             (default 1)
    "Mesh.SurfaceFaces": 1,   # shaded triangles                         (default 0)
    "Mesh.Lines": 1,          # the 12 meshed cube edges                 (default 0)
    "Mesh.LineWidth": 2,
    "Geometry.Points": 0,     # hide geometry entities: the 8 corner points and 12 curves
    "Geometry.Curves": 0,     # would otherwise draw on top of the mesh data
}
for option, value in DISPLAY.items():
    gmsh.option.setNumber(option, value)

# Tools > Options > Mesh toggles all of the above live; Mesh.NodeLabels = 1 prints tags.
if 'close' not in sys.argv:
    gmsh.fltk.run()

gmsh.finalize()


# Requested vs realized resolution

Prescribe a known target field, mesh it, then measure whether the mesh followed:

$$h_{\\text{target}}(x) = h_{\\min} + (h_{\\max}-h_{\\min})\\left[1 - \\exp\\left(-\\frac{\\lVert x - x_c\\rVert^2}{2\\sigma_h^2}\\right)\\right]$$

Small spacing at `xc`, growing to `hmax` far away. Per node:

$$r_i = \\frac{h^{\\text{mesh}}_i}{h^{\\text{target}}_i + \\varepsilon}, \\qquad \\delta_i = \\log r_i$$

`delta` near 0 means the realized local scale matches what was asked for. `h_mesh` is the
mean length of the domain edges incident to node *i*.

Two ways to hand the field to Gmsh, so a mismatch can be attributed:

- **matheval** — closed form as a Gmsh expression. No sampling error. Ground truth.
- **postview** — sampled on a background mesh, consumed as view data. Same mechanism a
  PDE error estimator will use later. Diverges from `matheval` when the background grid
  is too coarse to represent the well, which is sampling error, not Gmsh.


In [ ]:
import importlib

import numpy as np
import gmsh
import mesh_check, size_field
importlib.reload(mesh_check); importlib.reload(size_field)   # pick up edits to the .py files

if gmsh.isInitialized():
    gmsh.finalize()          # build() opens its own session; don't nest one inside another

# ---- edit these -----------------------------------------------------------
case = dict(hmin=0.01, hmax=0.10, sigma=0.15, xc=(0.5, 0.5, 0.0))
dim = 2                      # 3 -> unit cube, same code path
# ---------------------------------------------------------------------------

bounds = ([0, 0, 0], [1, 1, 1] if dim == 3 else [1, 1, 0])
for mode in ("matheval", "postview"):
    mesh = size_field.build(mode=mode, dim=dim, **case)
    d = size_field.diagnose(**mesh, **case)

    rep = mesh_check.check_mesh(**mesh, expected_bounds=bounds)
    bad = [k for k, v in rep["checks"].items() if not v["pass"]]
    print(f"{mode}:  {d['nodes']} nodes   median delta {d['median_delta']:+.3f}   "
          f"IQR {d['iqr_delta']:.3f}   median r {d['median_r']:.3f}   "
          f"mesh_check {'PASS' if rep['pass'] else 'FAIL ' + str(bad)}")

    q = np.quantile(d["delta"][np.isfinite(d["delta"])], [0.05, 0.25, 0.5, 0.75, 0.95])
    print("  delta at 5/25/50/75/95 pct:", "  ".join(f"{v:+.3f}" for v in q))
    print(f"  {'r':>6} {'n':>5} {'h_target':>9} {'h_mesh':>8} {'delta':>7}")
    for row in d["profile"]:
        print(f"  {row['r']:6.3f} {row['n']:5d} {row['h_target']:9.4f} "
              f"{row['h_mesh']:8.4f} {row['delta']:+7.3f}")
    print()


In [ ]:
# All five test cases, both modes. The hmin == hmax row is the control: a uniform target
# gives a uniform mesh, so its delta spread is the estimator's own noise floor -- anything
# wider in the graded rows is gradation, not a bug.
size_field.report(size_field.sweep(dim=dim))


## Look at the meshes

`size_field.show()` meshes every case in **one** Gmsh session, writes each to
`week1/meshes/<case>.msh`, and attaches the fields as post-processing views before handing
control to the GUI with the same `'close' not in sys.argv` guard as the mesh cell above.

Views, not models: Gmsh renders only the *current* model, so ten models in one window
would not be flippable. Views are listed and toggled independently, which is what lets you
compare cases without regenerating anything.

**Only views are drawn.** Gmsh keeps the *last* model meshed as current and would draw its
wireframe over everything (`Mesh.SurfaceEdges` is on by default) — and that last case is
`CASES[4]`, the uniform `hmin == hmax` control, so you would see an unrefined mesh sitting on
top of whichever case you were actually inspecting. `show()` turns all model and geometry
drawing off; each view supplies its own wireframe through `ShowElement`.

In the GUI:

- Left tree, **Post-processing** — one entry per `case x mode x field`, named
  `c0_matheval  delta   hmin 0.01  hmax 0.1  sigma 0.15`. All start hidden except the
  first; click to toggle. Flip between two `delta` views to see the difference directly.
- `h_target` is what you asked for, `h_mesh` what you got, `delta` the log ratio. On the
  `delta` view, colour away from 0 marks where the mesh disobeyed — the mid-field band
  showing up at about -0.09 is visible as a ring around `xc`.
- Element edges are drawn (`ShowElement`), so each view doubles as the wireframe.
- 3D (`dim = 3`): tets fill the volume and hide the interior. **Tools > Clipping** to cut in.

Thirty views with all five cases and three fields. Pass `fields=("delta",)` for ten.


In [ ]:
import importlib
import sys

import gmsh
import size_field
importlib.reload(size_field)

if gmsh.isInitialized():
    gmsh.finalize()          # show() opens its own session and closes it on the way out

dim = 2                      # 3 -> unit cube; use Tools > Clipping in the GUI to see inside

# Meshes every case, writes week1/meshes/*.msh, then blocks on the GUI until you close it.
# Pass run_gui=False to get the numbers without a window.
rows = size_field.show(dim=dim, fields=("h_target", "h_mesh", "delta"), outdir="meshes")

# Same table as the sweep cell, in the order the views are listed.
size_field.report([(label.split("_", 1)[1], case, d) for label, case, d in rows])
